# Explainable Fake Review Detection
### Genuine vs Human-Written Fake vs AI-Generated Fake Reviews

**Pipeline:** Auto-detect datasets → Merge → EDA → Text cleaning → TF-IDF → XGBoost → Evaluation → SHAP → Discussion

**Classes**
- `0` = Genuine
- `1` = Human Fake
- `2` = AI Fake

**No hardcoded file paths.** Add any/all of these as Kaggle "Input" datasets (or just
have them in the working directory if running elsewhere) and cell 1 will find and
classify them automatically by their columns:
- Hotel corpus (`deceptive`, `text` columns) → Genuine + Human Fake
- Professor's raw Amazon dataset (`reviewText`, `starRating` columns) → Genuine (cleaned first)
- Amazon OR/CG dataset (`text`/`text_` + `label` columns) → Genuine + AI Fake

Missing any of these just narrows which classes are available — the notebook always
runs with whatever it finds, rather than erroring out.


## 0. Setup

Kaggle notebooks usually come with `xgboost`, `shap`, and `nltk` pre-installed, so this
cell tries to just *use* them first. If something's missing, it attempts a `pip install`
/ NLTK download (needs internet); if that also fails, it falls back to alternatives that
work fully offline (Logistic Regression instead of XGBoost, sklearn's stopword list
instead of NLTK, no SHAP section) — so the rest of the notebook never gets stuck.

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, accuracy_score, f1_score)
from sklearn.preprocessing import label_binarize

sns.set_style("whitegrid")
RANDOM_STATE = 42

try:
    import xgboost as xgb
    HAVE_XGB = True
except ImportError:
    try:
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'xgboost'], check=True, timeout=60)
        import xgboost as xgb
        HAVE_XGB = True
    except Exception as e:
        print(f"xgboost unavailable ({e}). Will use Logistic Regression as the main model.")
        HAVE_XGB = False

try:
    import shap
    HAVE_SHAP = True
except ImportError:
    try:
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'shap'], check=True, timeout=60)
        import shap
        HAVE_SHAP = True
    except Exception as e:
        print(f"shap unavailable ({e}). Explainability section will be skipped.")
        HAVE_SHAP = False

try:
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    _ = lemmatizer.lemmatize('testing')  # forces wordnet to load, raises if missing
    HAVE_NLTK = True
except Exception:
    try:
        import nltk
        nltk.download('stopwords', quiet=True)
        nltk.download('wordnet', quiet=True)
        nltk.download('omw-1.4', quiet=True)
        from nltk.corpus import stopwords
        from nltk.stem import WordNetLemmatizer
        stop_words = set(stopwords.words('english'))
        lemmatizer = WordNetLemmatizer()
        _ = lemmatizer.lemmatize('testing')
        HAVE_NLTK = True
    except Exception as e:
        print(f"NLTK data unavailable ({e}). Using sklearn's stopword list, no lemmatization.")
        from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
        stop_words = set(ENGLISH_STOP_WORDS)
        lemmatizer = None
        HAVE_NLTK = False

print(f"xgboost: {HAVE_XGB}  |  shap: {HAVE_SHAP}  |  nltk stopwords+lemmatizer: {HAVE_NLTK}")


## 1. Auto-detect and load input datasets

Scans every CSV under `/kaggle/input/` (and the current directory) and classifies each
by its columns, so you never have to hardcode or guess a file path. Print the results
below and sanity-check that what got detected matches what you expect.

In [ ]:
def find_csvs():
    paths = glob.glob('/kaggle/input/**/*.csv', recursive=True) + glob.glob('*.csv')
    return sorted(set(paths))

def classify_csv(path):
    try:
        cols = set(c.lower() for c in pd.read_csv(path, nrows=5, low_memory=False).columns)
    except Exception:
        return None
    if {'deceptive', 'text'}.issubset(cols):
        return 'hotel'
    if {'reviewtext', 'starrating'}.issubset(cols):
        return 'amazon_raw_genuine'
    if ('text' in cols or 'text_' in cols) and 'label' in cols:
        return 'amazon_or_cg'
    return None

detected = {}
for p in find_csvs():
    kind = classify_csv(p)
    if kind and kind not in detected:
        detected[kind] = p
    print(f"{p}  ->  {kind}")

print("\nUsing:")
for k, v in detected.items():
    print(f"  {k}: {v}")
if not detected:
    print("  Nothing detected — add datasets via Kaggle's 'Add Input' panel, then re-run this cell.")


**Why the professor-provided Amazon dataset needs cleaning first**

The raw Amazon dataset has no fake/genuine label — it's just real reviews
(`reviewerId`, `reviewText`, `starRating`, `verifiedPurchase`, etc.), so it's used as
the **Genuine** class. Before that, three issues get handled:
1. **Column-shift corruption**: a small fraction of rows have text spilled into the
   `starRating` column (an unescaped comma somewhere in the original export). Rows
   where `starRating` isn't a valid 1–5 number are dropped, since we can't recover
   which field actually shifted.
2. **Duplicates / missing text** — dropped, standard cleaning.
3. **Near-empty reviews** (<3 characters) — dropped as noise, but short-but-real
   reviews ("Easy to use and very strong.") are kept.

In [ ]:
def clean_amazon_genuine(path):
    raw = pd.read_csv(path, low_memory=False)
    raw['starRating_numeric'] = pd.to_numeric(raw['starRating'], errors='coerce')
    before = len(raw)
    raw = raw[raw['starRating_numeric'].between(1, 5)]
    print(f"Dropped {before - len(raw)} corrupted rows (invalid starRating)")

    before = len(raw)
    raw = raw.dropna(subset=['reviewText']).drop_duplicates(subset=['reviewText'])
    print(f"Dropped {before - len(raw)} missing/duplicate reviewText rows")

    before = len(raw)
    raw = raw[raw['reviewText'].astype(str).str.len() >= 3]
    print(f"Dropped {before - len(raw)} near-empty reviews (<3 chars)")

    out = raw[['reviewText']].rename(columns={'reviewText': 'text'})
    out['label'] = 0  # Genuine
    return out

amazon_genuine = pd.DataFrame(columns=['text', 'label'])
if 'amazon_raw_genuine' in detected:
    amazon_genuine = clean_amazon_genuine(detected['amazon_raw_genuine'])
    print(f"\nClean professor-provided genuine reviews: {len(amazon_genuine)}")
else:
    print("No professor-provided Amazon file detected — skipping this source.")


**Why Genuine reviews combine multiple sources (hotel + Amazon)**

Human Fake only comes from the hotel domain. If Genuine were also hotel-only, the
model could learn to separate classes using domain vocabulary ("sounds like a hotel")
rather than genuine deception cues. Spanning two domains for Genuine closes that
shortcut and forces more general authenticity signals.

In [ ]:
hotel_genuine = pd.DataFrame(columns=['text', 'label'])
hotel_human_fake = pd.DataFrame(columns=['text', 'label'])
if 'hotel' in detected:
    hotel = pd.read_csv(detected['hotel'])
    hotel = hotel[['deceptive', 'text']].copy()
    hotel['label'] = hotel['deceptive'].map({'truthful': 0, 'deceptive': 1})
    hotel = hotel[['text', 'label']]
    hotel_genuine = hotel[hotel['label'] == 0]
    hotel_human_fake = hotel[hotel['label'] == 1]
    print(f"Hotel corpus: {len(hotel_genuine)} genuine, {len(hotel_human_fake)} human-fake")
else:
    print("No hotel corpus detected — Human Fake class will be empty unless another source is added.")

amazon_ai_fake = pd.DataFrame(columns=['text', 'label'])
amazon_or_genuine = pd.DataFrame(columns=['text', 'label'])
if 'amazon_or_cg' in detected:
    araw = pd.read_csv(detected['amazon_or_cg'])
    araw.columns = [c.lower() for c in araw.columns]
    text_col = 'text' if 'text' in araw.columns else 'text_'
    araw = araw[[text_col, 'label']].rename(columns={text_col: 'text'})
    # Normalize whatever label scheme is used (OR/CG strings, or 0/1 ints) to
    # 0=Genuine / 2=AI-Fake, matching this notebook's class scheme.
    if araw['label'].dtype == object:
        araw['label'] = araw['label'].map({'OR': 0, 'CG': 2})
    else:
        araw['label'] = araw['label'].map({0: 0, 1: 2})
    amazon_or_genuine = araw[araw['label'] == 0]
    amazon_ai_fake = araw[araw['label'] == 2]
    print(f"Amazon OR/CG dataset: {len(amazon_or_genuine)} genuine, {len(amazon_ai_fake)} AI-fake")
else:
    print("No Amazon OR/CG dataset detected — AI Fake class will be empty (2-class mode).")

genuine_combined = pd.concat([hotel_genuine, amazon_genuine, amazon_or_genuine], ignore_index=True)
human_fake_combined = hotel_human_fake
ai_fake_combined = amazon_ai_fake

parts = []
class_names = {}
if len(genuine_combined) > 0:
    class_names[0] = 'Genuine'
    parts.append(genuine_combined.assign(label=0))
if len(human_fake_combined) > 0:
    class_names[1] = 'Human Fake'
    parts.append(human_fake_combined.assign(label=1))
if len(ai_fake_combined) > 0:
    class_names[2] = 'AI Fake'
    parts.append(ai_fake_combined.assign(label=2))

assert parts, "No usable data detected at all — add at least one dataset via Kaggle's Add Input panel."

df = pd.concat(parts, ignore_index=True)
df['class_name'] = df['label'].map(class_names)
df = df.dropna(subset=['text']).reset_index(drop=True)
n_classes = df['label'].nunique()
print(f"\n{n_classes}-class mode. Final dataset:")
print(df['class_name'].value_counts())
df.head()


## 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='class_name', order=list(class_names.values()))
plt.title('Class Distribution')
plt.xlabel('')
plt.tight_layout()
plt.show()

vc = df['class_name'].value_counts()
print(vc)
print(f"\nClass imbalance ratio (max/min): {vc.max() / vc.min():.2f}")


In [ ]:
df['word_count'] = df['text'].astype(str).str.split().apply(len)

plt.figure(figsize=(7,4))
sns.histplot(data=df, x='word_count', hue='class_name', bins=30, kde=True, element='step')
plt.title('Review Length (word count) by Class')
plt.xlabel('Word count')
plt.xlim(0, df['word_count'].quantile(0.99))
plt.tight_layout()
plt.show()

df.groupby('class_name')['word_count'].describe()


In [ ]:
# Top words per class (raw, before cleaning — sanity check for obvious artifacts)
for cls in df['class_name'].unique():
    texts = df.loc[df['class_name'] == cls, 'text'].astype(str)
    cv = CountVectorizer(stop_words='english', max_features=15)
    X = cv.fit_transform(texts)
    freqs = np.asarray(X.sum(axis=0)).flatten()
    words = cv.get_feature_names_out()
    top = sorted(zip(words, freqs), key=lambda x: -x[1])
    print(f"\nTop words — {cls}:")
    for w, f in top:
        print(f"  {w}: {f}")


**What to look for here before moving on:**
- Are classes balanced? If not, note it — you'll need `class_weight` / stratified sampling later.
- Do "AI Fake" reviews look noticeably more generic/polished, matching the expected finding
  from your project summary?
- Any obvious domain-leakage words worth flagging as a limitation later.

## 3. Text preprocessing
Lowercase → remove punctuation/digits → remove stopwords → lemmatize (if NLTK data was
available in cell 0 — otherwise stopword removal still runs, just without lemmatization).

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r"\d+", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    if lemmatizer is not None:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head()


## 4. Train/test split + TF-IDF features
Split BEFORE vectorizing, and fit the vectorizer only on the training set, to avoid leaking test-set vocabulary into training.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'], df['label'],
    test_size=0.2, random_state=RANDOM_STATE, stratify=df['label']
)

tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1,2), min_df=3)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


## 5. Train models

XGBoost as the main model when available; Logistic Regression always runs too, as a
baseline (or as the main model, if XGBoost isn't installed).

**Class imbalance note:** Genuine substantially outnumbers Human Fake in this
configuration. Both models use class balancing (`class_weight='balanced'` for
Logistic Regression, `scale_pos_weight` for binary XGBoost) so they don't just
collapse to predicting the majority class — and Section 6 reports balanced-accuracy,
Matthews correlation coefficient (MCC) and PR-AUC alongside accuracy, since plain
accuracy is misleading under heavy imbalance (a classifier that always predicts
Genuine would already score highly on accuracy alone).

In [ ]:
main_model = None
pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)  # only meaningful for 2-class

if HAVE_XGB:
    xgb_kwargs = dict(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        random_state=RANDOM_STATE
    )
    if n_classes > 2:
        xgb_kwargs.update(objective='multi:softprob', eval_metric='mlogloss')
    else:
        xgb_kwargs.update(objective='binary:logistic', eval_metric='logloss', scale_pos_weight=pos_weight)
    main_model = xgb.XGBClassifier(**xgb_kwargs)
    main_model.fit(X_train, y_train)
    main_model_name = "XGBoost"

baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')
baseline.fit(X_train, y_train)

if main_model is None:
    main_model = baseline
    main_model_name = "Logistic Regression (main model — XGBoost unavailable)"

print(f"Trained. Main model: {main_model_name}")
if n_classes == 2:
    print(f"Class balance -> scale_pos_weight used for XGBoost: {pos_weight:.2f}")


## 6. Evaluation

In [ ]:
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef, average_precision_score

def evaluate(model, name):
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)

    print(f"=== {name} ===")
    print(classification_report(y_test, preds, target_names=list(class_names.values())))

    bal_acc = balanced_accuracy_score(y_test, preds)
    mcc = matthews_corrcoef(y_test, preds)
    print(f"Balanced accuracy: {bal_acc:.4f}")
    print(f"Matthews correlation coefficient: {mcc:.4f}")

    if n_classes > 2:
        y_test_bin = label_binarize(y_test, classes=sorted(class_names.keys()))
        auc = roc_auc_score(y_test_bin, proba, average='macro', multi_class='ovr')
        pr_auc = average_precision_score(y_test_bin, proba, average='macro')
    else:
        auc = roc_auc_score(y_test, proba[:, 1])
        pr_auc = average_precision_score(y_test, proba[:, 1])
    print(f"Macro ROC-AUC: {auc:.4f}")
    print(f"Macro PR-AUC: {pr_auc:.4f}  (more informative than ROC-AUC under heavy class imbalance)")

    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(class_names.values()),
                yticklabels=list(class_names.values()))
    plt.title(f'Confusion Matrix — {name}')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

    return preds, proba

main_preds, main_proba = evaluate(main_model, main_model_name)
if main_model is not baseline:
    _ = evaluate(baseline, "Logistic Regression (baseline)")


## 7. SHAP explainability
Identifies which words/phrases pushed the model toward each class. Only runs if `shap`
and XGBoost were both available in cell 0 — SHAP's TreeExplainer needs a tree model.

In [ ]:
if HAVE_SHAP and HAVE_XGB and main_model_name == "XGBoost":
    explainer = shap.TreeExplainer(main_model)
    sample_idx = np.random.choice(X_test.shape[0], size=min(200, X_test.shape[0]), replace=False)
    X_test_sample = X_test[sample_idx]
    shap_values = explainer.shap_values(X_test_sample)
    feature_names = tfidf.get_feature_names_out()
    print("SHAP values computed")
else:
    print("Skipping SHAP — needs both shap and xgboost installed and XGBoost as the main model.")


In [ ]:
if HAVE_SHAP and HAVE_XGB and main_model_name == "XGBoost":
    if n_classes > 2:
        for i, cname in class_names.items():
            plt.figure()
            shap.summary_plot(shap_values[i], X_test_sample, feature_names=feature_names,
                               max_display=15, show=False, plot_type='bar')
            plt.title(f'Top SHAP features — {cname}')
            plt.tight_layout()
            plt.show()
    else:
        shap.summary_plot(shap_values, X_test_sample, feature_names=feature_names, max_display=15)


## 8. Sample-size sensitivity — does more Genuine data actually help?

Human Fake is fixed at 800 rows (that's the entire hotel corpus) no matter how much
Genuine data gets added. So a natural question: is it worth using the professor's
*full* multi-million-row Amazon file, or does performance plateau well before that?

**This cell answers that directly.** It re-runs the identical pipeline at three
Genuine sample sizes (2,000 / 8,000 / however much Genuine data is currently loaded,
up to ~19,000), keeping Human Fake fixed throughout, and plots balanced accuracy, MCC
and PR-AUC — the imbalance-aware metrics — against sample size. Plain accuracy and
ROC-AUC are deliberately left out of this comparison, since they stay misleadingly
high at every sample size under this level of imbalance.

If the metrics **plateau** rather than keep climbing, that's direct evidence that
your current sample size is already sufficient, and processing millions more Genuine
rows would cost far more compute for negligible benefit — because the bottleneck
isn't Genuine data volume, it's the fixed 800-row Human Fake class, which no amount
of additional Genuine data can address.

In [ ]:
def run_at_sample_size(n_genuine_target, seed=RANDOM_STATE):
    genuine_pool = genuine_combined.assign(label=0)
    n_available = len(genuine_pool)
    n_use = min(n_genuine_target, n_available)
    genuine_sample = genuine_pool.sample(n=n_use, random_state=seed)

    sub = pd.concat([genuine_sample, human_fake_combined.assign(label=1)], ignore_index=True)
    sub['clean_text'] = sub['text'].apply(clean_text)

    Xtr_t3, Xte_t3, ytr3, yte3 = train_test_split(
        sub['clean_text'], sub['label'], test_size=0.2, random_state=seed, stratify=sub['label']
    )
    tfidf3 = TfidfVectorizer(max_features=8000, ngram_range=(1,2), min_df=3)
    Xtr3 = tfidf3.fit_transform(Xtr_t3)
    Xte3 = tfidf3.transform(Xte_t3)

    pw = (ytr3 == 0).sum() / max((ytr3 == 1).sum(), 1)
    if HAVE_XGB:
        m = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                               objective='binary:logistic', eval_metric='logloss',
                               scale_pos_weight=pw, random_state=seed)
    else:
        m = LogisticRegression(max_iter=1000, random_state=seed, class_weight='balanced')
    m.fit(Xtr3, ytr3)
    p3 = m.predict(Xte3)
    proba3 = m.predict_proba(Xte3)[:, 1]

    return {
        'n_genuine': n_use,
        'n_total': len(sub),
        'balanced_accuracy': balanced_accuracy_score(yte3, p3),
        'mcc': matthews_corrcoef(yte3, p3),
        'pr_auc': average_precision_score(yte3, proba3),
        'human_fake_f1': classification_report(yte3, p3, output_dict=True)['1']['f1-score'],
    }

if 'hotel' in detected and len(genuine_combined) > 0:
    sample_sizes = sorted(set([2000, 8000, len(genuine_combined)]))
    sensitivity_results = [run_at_sample_size(n) for n in sample_sizes]
    sensitivity_df = pd.DataFrame(sensitivity_results)
    print(sensitivity_df)

    fig, ax = plt.subplots(figsize=(7,4.5))
    ax.plot(sensitivity_df['n_genuine'], sensitivity_df['balanced_accuracy'], marker='o', label='Balanced accuracy')
    ax.plot(sensitivity_df['n_genuine'], sensitivity_df['mcc'], marker='o', label='MCC')
    ax.plot(sensitivity_df['n_genuine'], sensitivity_df['pr_auc'], marker='o', label='PR-AUC')
    ax.plot(sensitivity_df['n_genuine'], sensitivity_df['human_fake_f1'], marker='o', label='Human Fake F1')
    ax.set_xlabel('Genuine sample size')
    ax.set_ylabel('Score')
    ax.set_title('Performance vs. Genuine sample size (Human Fake fixed at 800)')
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print("Need both hotel and Genuine data loaded to run this comparison.")


**How to read this for your report/presentation:** plot the lines from the chart
above. If they're roughly flat from 8,000 to ~19,000, say explicitly that you tested
whether scaling to the full multi-million-row file was worth the computational cost,
found performance had already plateaued, and therefore used the ~19,000-row sample as
your reported result — that is a stronger, more defensible methodological statement
than either arbitrarily picking one sample size or claiming "more data is always
better" without evidence.

## 9. Domain-leakage check — hotel-only comparison

The mixed-domain run above (hotel + Amazon combined into Genuine) got strong metrics,
but SHAP showed the model leaning heavily on domain words ("hotel", "chicago", "room")
rather than deception style, and Genuine/Human-Fake reviews had very different length
distributions. Both are shortcuts a model could exploit without learning anything about
actual deception.

**This cell isolates that.** It reruns the identical pipeline (same cleaning, same
TF-IDF settings, same model type) using data from a **single domain only** — hotel
`truthful` vs hotel `deceptive`, both written for the same task, both similar length.
If accuracy stays close to the mixed-domain run, that's evidence the model is picking
up on genuine deception style. If accuracy drops noticeably, that gap is a rough
estimate of how much of the original 92% was actually just domain/length leakage.

In [ ]:
if 'hotel' in detected:
    hotel_only = pd.concat([
        hotel_genuine.assign(label=0),
        hotel_human_fake.assign(label=1)
    ], ignore_index=True)
    hotel_only['clean_text'] = hotel_only['text'].apply(clean_text)

    Xtr_t2, Xte_t2, ytr2, yte2 = train_test_split(
        hotel_only['clean_text'], hotel_only['label'],
        test_size=0.2, random_state=RANDOM_STATE, stratify=hotel_only['label']
    )
    tfidf2 = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2)
    Xtr2 = tfidf2.fit_transform(Xtr_t2)
    Xte2 = tfidf2.transform(Xte_t2)

    if HAVE_XGB:
        hotel_model = xgb.XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            objective='binary:logistic', eval_metric='logloss',
            random_state=RANDOM_STATE
        )
    else:
        hotel_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    hotel_model.fit(Xtr2, ytr2)

    preds2 = hotel_model.predict(Xte2)
    proba2 = hotel_model.predict_proba(Xte2)[:, 1]

    print("=== Hotel-only (same-domain) result ===")
    print(classification_report(yte2, preds2, target_names=['Genuine (hotel)', 'Human Fake (hotel)']))
    print(f"ROC-AUC: {roc_auc_score(yte2, proba2):.4f}")

    cm2 = confusion_matrix(yte2, preds2)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm2, annot=True, fmt='d', cmap='Oranges',
                xticklabels=['Genuine', 'Human Fake'], yticklabels=['Genuine', 'Human Fake'])
    plt.title('Confusion Matrix — Hotel-only (same domain)')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

    mixed_acc = accuracy_score(y_test, main_preds)
    same_domain_acc = accuracy_score(yte2, preds2)
    gap = mixed_acc - same_domain_acc
    print(f"\nMixed-domain accuracy: {mixed_acc:.4f}")
    print(f"Same-domain accuracy:  {same_domain_acc:.4f}")
    print(f"Gap: {gap:+.4f}  "
          f"({'mixed-domain result likely inflated by domain leakage' if gap > 0.03 else 'gap is small — leakage concern is minor'})")
else:
    print("No hotel dataset detected — cannot run the same-domain comparison.")


**How to read this for your dissertation:**
- If same-domain accuracy is close to the mixed-domain accuracy → deception style itself
  carries real signal, and the earlier 92% is reasonably trustworthy.
- If same-domain accuracy drops noticeably → a meaningful chunk of the original 92% came
  from the model exploiting domain vocabulary and length differences, not genuine
  deception cues. Report the honest, same-domain number as your headline result, and
  discuss the mixed-domain result as a caveat/limitation rather than the main finding.
- Either way, this comparison is a legitimate methodological contribution for your
  discussion section — it shows awareness of confounding variables, which is exactly
  what markers look for.

## 10. Cross-domain generalization check — hotel model tested on Amazon data

Section 9 trained and tested a model using only hotel data. This section goes further:
it takes that **same hotel-only model** and tests it on the Amazon Genuine reviews it
has never seen, from a completely different domain.

**Important limitation up front**: the Amazon data has no confirmed Human Fake label,
so this is *not* an accuracy test — there's nothing to check predictions against. What
it actually tells you:

1. **Generalization sanity check** — does the model behave sensibly on new-domain
   text, or does it just collapse into predicting one class for everything?
2. **A rough contamination probe** — if you read the earlier limitation about the
   Amazon Review Manipulation Dataset possibly containing leftover fake reviews from
   the original attack campaign (Kaghazgaran, Alfifi and Caverlee, 2019), the reviews
   this model flags as "Human Fake" are your best candidates for which rows those
   might be. This is suggestive, not proof — the hotel-trained model was never
   designed to catch Amazon-style fake reviews specifically.

In [ ]:
if 'hotel' in detected and len(genuine_combined) > 0 and 'amazon_genuine' in dir() and len(amazon_genuine) > 0:
    amazon_only = amazon_genuine.copy()
    amazon_only['clean_text'] = amazon_only['text'].apply(clean_text)
    X_amazon_cross = tfidf2.transform(amazon_only['clean_text'])  # hotel-only vectorizer from Section 9

    preds_cross = hotel_model.predict(X_amazon_cross)
    proba_cross = hotel_model.predict_proba(X_amazon_cross)[:, 1]

    n_genuine_pred = (preds_cross == 0).sum()
    n_fake_pred = (preds_cross == 1).sum()
    print(f"Amazon reviews tested: {len(amazon_only)}")
    print(f"Predicted Genuine:    {n_genuine_pred} ({n_genuine_pred/len(amazon_only)*100:.1f}%)")
    print(f"Predicted Human Fake: {n_fake_pred} ({n_fake_pred/len(amazon_only)*100:.1f}%)")
    print(f"Mean fake-probability: {proba_cross.mean():.4f}")

    plt.figure(figsize=(7,4))
    sns.histplot(proba_cross, bins=30)
    plt.axvline(0.5, color='red', linestyle='--', label='Decision threshold')
    plt.xlabel('Predicted probability of Human Fake')
    plt.title('Hotel-trained model\'s fake-probability scores on Amazon reviews')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Vocabulary overlap: how much hotel vocabulary actually appears in Amazon text
    nonzero_per_doc = (X_amazon_cross > 0).sum(axis=1)
    zero_vocab = (nonzero_per_doc == 0).sum()
    print(f"\nAmazon reviews with ZERO matching hotel-model vocabulary: {zero_vocab} ({zero_vocab/len(amazon_only)*100:.1f}%)")

    # Save the most-suspicious rows as candidates for manual inspection
    amazon_only['fake_probability'] = proba_cross
    most_suspicious = amazon_only.sort_values('fake_probability', ascending=False).head(15)
    print("\nTop 15 Amazon reviews the hotel-trained model considers most 'Human Fake'-like:")
    print("(candidates for manual inspection re: possible attack contamination)")
    for _, row in most_suspicious.iterrows():
        print(f"  [{row['fake_probability']:.3f}] {row['text'][:100]}")
else:
    print("Need hotel model (Section 9) and Amazon genuine data loaded to run this check.")


**How to read this for your report:** report the Genuine/Human-Fake split honestly
as a generalization observation, not an accuracy figure — be explicit that there is no
ground truth on the Amazon side. If you want to go further, you could manually read a
sample of the top "most suspicious" rows printed above and note whether any look like
plausible planted fake reviews (generic, overly promotional, disconnected from specific
product details) versus normal customer reviews — but frame this as a qualitative,
illustrative check, not a validated finding.

## 11. Discussion & conclusion (fill in after reviewing your outputs)

Prompts to guide your write-up, tied back to your research question:

- **Does SHAP (or, if unavailable, the top-words EDA from step 2) support the expected
  patterns from your project summary?** (Genuine → personal/specific experience words;
  Human Fake → exaggerated praise/superlatives; AI Fake → generic, overly polished
  phrasing.) Quote 2–3 supporting words per class as evidence.
- **Where does the model struggle?** Look at the confusion matrix — which class pairs
  get confused most, and does that make conceptual sense?
- **Domain leakage caveat**: since different classes partly come from different review
  domains (hotels vs Amazon products), some "signal" may be domain vocabulary rather
  than genuine deception cues — worth an explicit limitation paragraph.
- **How does the main model compare to the baseline?** If the gap is small, that itself
  is a finding worth discussing.
- **Sample-size finding**: does Section 8 show a plateau? State the sample size you
  settled on and justify it using that evidence, rather than justifying it by "this is
  what I had time to run."
- **Cross-domain generalization (Section 10)**: what fraction of Amazon reviews did the
  hotel-only model flag as suspicious? Discuss this as a generalization signal, and
  optionally as weak evidence about possible contamination in the Amazon dataset,
  being careful not to overclaim given there's no ground truth to confirm it against.
